FRAME TO FRAME INFERENCE & EVALUATION

In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
import pickle
import warnings
import gzip
import scipy.io
from scipy import signal
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

# 8k Net trained
# workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
workspace_dir = '/home/adelval/BTS/TFM/afterburner8k_win20/'
# 16k Net trained
# workspace_dir = '/home/adelval/BTS/TFM/test/'

reduced_net = False

sys.path.append(workspace_dir + 'src/net1')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [2]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

PREEMPHASIS OPTIMIZATION

In [3]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

def optimized_preemphasis(x, alpha=0.97):
    x = np.array(x, dtype=np.float64)
    x[1:] = x[1:] - alpha * x[:-1]
    x[0] = x[0] * (1 - alpha)
    return x

# Divide x into overlapping frames of fixed length without extending to slide last frame
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010, Mw=20):
    # Mw limit the number of windows
    N = int(Ns * fs)            # Number of samples in each window 640
    M = int(Ms * fs)            # Step size (number of samples between window starts) 160
    n = (len(x) + M - 1) // M   # Number of frames 23
    # print("Number of frames", n)    
    # T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    # Se ignora el padding porque la ventana se deslizará
    # if T > len(x):
    #     print("rellena con ceros")
    #     xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, Mw*M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    xa = xa[ind.astype(int).T].astype(np.float32)

    return xa

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT):
    Xfreq = np.abs(np.fft.fft(X, NFFT)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:,:(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals

def window_fft(x, hamming_win, nfft):
    x = offset(x)
    x = optimized_preemphasis(x)

    return fft(x*hamming_win, nfft)


def log_psd(Xfft, eps=1e-8):
    """
    Computes the log-scaled power spectral density (PSD) from the FFT output.
    
    Parameters:
        Xfft (np.ndarray): FFT output, shape (N, nfft//2)
        scale (float): Multiplier for log scale, typically 2.0 for power (or 10/20 for dB)
        eps (float): Small constant to avoid log(0)
    
    Returns:
        np.ndarray: Log-scaled PSD
    """
    Xfft = np.asarray(Xfft, dtype=np.float32)
    X_log_psd = 2 * np.log10(np.abs(Xfft) + eps)
    return X_log_psd


def frame_fft(data, fs, w, nfft, max_windows, hamming_win):
    

    pre_offset = time.time()
    x = offset(data)
    global accum_offset 
    accum_offset += (time.time()-pre_offset)

    # Emphasis to increase the amplitude of high freq
    pre_emphasis = time.time()
    x = optimized_preemphasis(x)
    # print(f"Preemphasis time: {(time.time()-pre_emphasis)*1000} ms")
    global accum_preemphasis 
    accum_preemphasis += (time.time()-pre_emphasis)

    X = windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows) * hamming_win

    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft)  # Instead of calling a funtion, do directly

    # print(f"la shape de Xfft es {Xfft.shape}")
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    
    # eps=1e-8
    # X = np.log10(X + eps)  # cambiar el producto por 20 que e slo mismo que hacer el producto
    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    # print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [4]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb



def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b



def frame_fb_mfcc(data, fs, B, w, nfft, max_windows, fb, dct, hamming_win):
    x = offset(data)
    x = optimized_preemphasis(x)
    
    X = windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows) * hamming_win

    Xfft = fft(X, nfft)
    Xb = np.log(Xfft.dot( fb[0] ) + 1)
    Xc = Xb.dot(dct[0])                          
    
    X = np.concatenate( [Xb, Xc], 1 )
    
    X = np.asarray(X, dtype=np.float32)
    
    return X

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc, mu, std):
    # Normalization of fbmfcc
    frame_fbmfcc -= mu
    frame_fbmfcc /= std + 1e-6

    return frame_fbmfcc
    

LOADS FOR WINDOWS NET

In [5]:
# Load model dimensions and weights for windowing

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)


input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 


print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))

# Load net for windowing
sys.path.append( workspace_dir + 'src/net1')
from net_snr import Net_snr 
net_snr = Net_snr(input_dim, output_dim, cuda=True, single_gpu=True)
net_snr.load( workspace_dir + 'data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:


    nb_params: 29.99M
    cuda: True
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    restoring epoch: 500
    restoring opt: adama, lr: 0.000019
    opt: adama, 1.94812e-05, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 1.9481214405699424e-05
    weight_decay: 0
)
    reading /home/adelval/BTS/TFM/afterburner8k_win20/data/model/theta_last


500

In [6]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    n_frames, fft_fb = x.shape
    x = x.reshape(1, n_frames, fft_fb)
    #x = x.unsqueeze(0) # Batch dimension

    snr = net_snr.predict(x)
    snr = to_numpy(snr.squeeze())
    
    #scipy.io.savemat(f, mdict={'snr': snr})
    # x = to_numpy(x.squeeze())
    # snr = to_numpy(snr.squeeze())
    return snr


In [7]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    # print(f'el tipo de yw es {yw.dtype}')
    # print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    #it = int(np.floor((data.size-frame)/shift))
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        # print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        # print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        # print(Xfft.shape)
        # print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        # print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        # print(f'El tamaño de la salida del filtro sera {outf.shape}')
        # print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        # print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        # print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # print(f'El tipo de la señal es {data.dtype}')
    # print(f'El tipo de la red es {snr_net.dtype}')
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    # print(f'El tipo de la señal es {data.dtype} con ruido añadido')
    snr_net = snr_net.reshape(-1,1) # para darle 2-D
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    # print(f'El filtro será {filt.dtype}')
    # print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


In [8]:
def fft(X, NFFT):
    Xfreq = np.abs(np.fft.fft(X, NFFT)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


def window_fft(x, hamming_win, nfft):
    x = offset(x)
    x = optimized_preemphasis(x) # varía el resultado al aplicarlo solo a una ventana sin contexto
    x = x*hamming_win

    return fft(x, nfft)

def log_psd(Xfft, eps=1e-8):
    """
    Computes the log-scaled power spectral density (PSD) from the FFT output.
    
    Parameters:
        Xfft (np.ndarray): FFT output, shape (N, nfft//2)
        scale (float): Multiplier for log scale, typically 2.0 for power (or 10/20 for dB)
        eps (float): Small constant to avoid log(0)
    
    Returns:
        np.ndarray: Log-scaled PSD
    """
    Xfft = np.asarray(Xfft, dtype=np.float32)
    X_log_psd = 2 * np.log10(np.abs(Xfft) + eps)
    return X_log_psd


def frame_fb_mfcc_2(Xfft, fb, dct):

    Xb = np.log(Xfft.dot( fb ) + 1)
    Xc = Xb.dot(dct)                          

    X = np.concatenate( [Xb, Xc] )
    
    X = np.asarray(X, dtype=np.float32)

    X -= mu
    X /= std + 1e-6
    
    return X

In [28]:
# Parámetros
fs=8000
B=32
w=[0.040]
m=0.01
nfft=1024
gmin = 0.0562
min_windows = 4
max_windows = 20
diezmation_factor = 2

# Cálculo de los filtros
N =[int(wi * fs) for wi in w]
F = int(nfft/2)
fb_time = time.time()
fb = fb_etsi(F, B, fs)
print(f'El tiempo de cálculo de los filtros es {(time.time() - fb_time)*1000} ms')
dct_time = time.time()
dct = f_base_dct(B)
print(f'El tiempo de cálculo de las bases dct es {(time.time() - dct_time)*1000} ms')

# Media y desviación para la normalización
file = workspace_dir + 'data/model/fe1_norm1.pkl'
mu, std = read_pkl(file)


# Ventana de hamming
hamming_win = np.hamming(fs * w[0])

# Variables globales para el cálculo de los tiempos
accum_offset = 0
accum_preemphasis = 0
accum_windowing = 0
accum_fft = 0
accum_log = 0
accum_fb_mfcc = 0
accum_norm = 0 
accum_inf = 0
diff_acum = 0
diff_inf_acum = 0


## SELECCIÓN DE AUDIO
# AUDIOS 16K
# x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/livekit/audio_received.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner_2023/afterburner16k/data/audio/callcenter_movistar/AUDIOS/Audio-AudioModule_708351_AudioChannel_22403522_19-May-2023_15.53.49.097.wav']

# AUDIOS 8K
x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/11-CH0_C01_construction_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/36635c2b-ca8a-5d7c-abb6-42981f7613e9.wav']

print(f'El audio elegido es {x_test[0]}')

if(reduced_net):
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16_opt.wav')
    print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
else:
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16_opt.wav')
    # print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
    print(f'El audio mejorado es {output_enh_int} de tipo int16')



print(f'\n|-----------------------------INITIAL PARAMETERS-----------------------------|')
audio, fs = read_audio(x_test[0])
print(f'  Frecuencia de muestreo: {fs} Hz')
print(f'  Duración del audio: {len(audio)/fs} s - {len(audio)} samples')
frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
print(f'  Tamaño de frame: {frame_samples} samples')
shift_size = m  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
print(f'  Desplazamiento: {shift_samples} samples')
window_size = w[0]  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640
print(f'  Tamaño de ventana: {window_samples} samples')
window_inference_min = int((min_windows + (window_size/frame_size)-1) * frame_samples)
window_inference_max = int((max_windows + (window_size/frame_size)-1) * frame_samples)
print(f'  Buffer progresivo: {window_inference_min} - {window_inference_max} samples')
buffer_frame = np.zeros(0)  # Buffer de ventana recibida
Xfft_windows_list = []
print(f'|----------------------------------------------------------------------------|\n')

it = 0
snr_frame_mask = np.ones((512,min_windows)) # Inicializado con la duración de la ventana de inferencia
yenh = np.zeros(len(audio)) # Inicializado con la duración del audio original

# CALCULO DE LA MÁSCARA SNR 
for n_frame in range(int((len(audio)/fs)*100)):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]

    # print(f'\n{n_frame} frame de {len(frame)} --> {frame[:10]}')
    buffer_frame = np.concatenate([buffer_frame, frame])
    
    start_time = time.time()

    if len(buffer_frame) >= window_samples:
        work_window = buffer_frame[:window_samples]

        start_fft = time.time()
        Xfft = window_fft(work_window, hamming_win, nfft)
        accum_fft += (time.time()-start_fft)

        start_log = time.time()
        log_psd_Xfft = log_psd(Xfft)
        accum_log += (time.time()-start_log)

        start_fb = time.time()
        fb_windows = frame_fb_mfcc_2(Xfft, fb, dct)
        accum_fb_mfcc += (time.time()-start_fb)

        windows_concat = np.concatenate((log_psd_Xfft,fb_windows))
        Xfft_windows_list.append(windows_concat)

        if n_frame < max_windows:   
                if n_frame >= min_windows:
                    # print("Reached MIN WINDOW --> STRATING INFERENCE")
                    transformed_windows = np.vstack(Xfft_windows_list)

                    start_prof = time.time()
                    if(n_frame % diezmation_factor == 0):
                        snr_frame_mask = net_eval(transformed_windows)
                        snr_frame_mask = snr_frame_mask.T
                        print(f'Time of inference pre: {(time.time()-start_prof)*1000} ms')
                    accum_inf += (time.time()-start_prof)

        # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
        else:
            Xfft_windows_list.pop(0)
            transformed_windows = np.vstack(Xfft_windows_list)
            start_prof = time.time()
            # Process inference with factor 2 diezmation
            if(n_frame % diezmation_factor == 0):
                snr_frame_mask = net_eval(transformed_windows)
                snr_frame_mask = snr_frame_mask.T
                print(f'Time of inference: {(time.time()-start_prof)*1000} ms')
            
            accum_inf += (time.time()-start_prof)

        buffer_frame = buffer_frame[shift_samples:]

        # Aqui haría la evaluacion con la máscara pertinente (para las primeras 3 ventanas sin máscara calculada)
        # cnt = int(n_frame - w[0]/m) Ajustar al tamaño de la ventana
        cnt = n_frame - 3
        # print(f'EVALUATION OF WINDOW {cnt}')
        x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples

        xenh, filt = noiseReduction(x, snr_frame_mask[:,-1], fs, window_samples, shift_samples, nfft, gmin)

        slice_size = min(len(yenh) - cnt * shift_samples, window_samples)

        yenh[cnt * shift_samples : cnt * shift_samples + slice_size] += xenh[0:slice_size]


    end_time = time.time()
    diff_acum += end_time - start_time
    # print(f'Tiempo de procesamiento del frame {n_frame} completo es de {(end_time-start_time)*1000} ms')

print(f'\n|-----------------------------FINAL TIME STATS-------------------------------|')
print(f'  El tiempo medio de procesamiento de la FFT es de {(accum_fft/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la escala log es de {(accum_log/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la FB MFCC es de {(accum_fb_mfcc/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la inferencia es de {(accum_inf/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento total es de {(diff_acum/n_frame)*1000} ms')
print(f'|----------------------------------------------------------------------------|\n')


yenh = yenh/3     # 4 because in OverLapAdd we sum 4 times the frame

yenh = np.array(yenh*(2 ** 15), dtype=np.int16)     # set int16 wav format

wavfile.write(output_enh_int,fs,yenh)


#--------------------------SNR POST--------------------------#
# post_vad = compute_vad(yenh_clipped, fs, w[0], m, nfft[0])
# snr_post = int(wada_snr(yenh_clipped, fs, pre_vad))
# print('snr(wada)=%idB, file: %s' % (snr_post, output_enh))
#------------------------------------------------------------#




El tiempo de cálculo de los filtros es 1.1763572692871094 ms
El tiempo de cálculo de las bases dct es 1.2633800506591797 ms
El audio elegido es /home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav
El audio mejorado es /home/adelval/BTS/TFM/audios/enh/5-CH0_C01_stadium_15dB_f2f_enh_HM_2_8k_4to20w_40ms_s10_int16_opt.wav de tipo int16

|-----------------------------INITIAL PARAMETERS-----------------------------|
  Frecuencia de muestreo: 8000 Hz
  Duración del audio: 3.374125 s - 26993 samples
  Tamaño de frame: 80 samples
  Desplazamiento: 80 samples
  Tamaño de ventana: 320 samples
  Buffer progresivo: 560 - 1840 samples
|----------------------------------------------------------------------------|

Time of inference pre: 40.625810623168945 ms
Time of inference pre: 21.63219451904297 ms
Time of inference pre: 21.683931350708008 ms
Time of inference pre: 21.287202835083008 ms
Time of inference pre: 21.71158790588379 ms
Time of inference pre: 21.492004394

ONNX INFERENCE


In [15]:
import onnx
import onnxruntime


onnx_model = onnx.load("/home/adelval/BTS/TFM/afterburner8k_win20/data/model/net_snr.onnx")
onnx.checker.check_model(onnx_model)

In [13]:
import onnx
from onnxsim import simplify

onnx_simplified, check = simplify(onnx_model)

assert check, "Simplified ONNX model could not be validated"

with open("/home/adelval/BTS/TFM/afterburner8k_win20/data/model/net_snr.onnx", "wb") as f:
    f.write(onnx_simplified.SerializeToString())
# onnx.save(onnx_simplified, onnx_model)  # overwrite


In [16]:

ort_session = onnxruntime.InferenceSession("/home/adelval/BTS/TFM/afterburner8k_win20/data/model/net_snr.onnx", providers=["CUDAExecutionProvider"])
# ort_session = onnxruntime.InferenceSession("/home/adelval/BTS/TFM/afterburner8k_win20/data/model/net_snr.onnx", providers=['TensorrtExecutionProvider'])

In [14]:
assert 'CUDAExecutionProvider' in onnxruntime.get_available_providers()
device_name = 'gpu'

optimized_model_filepath = os.path.join(output_dir, "optimized{}_model_{}.onnx".format(device_name, 4))
print("Optimized model saved to: ", optimized_model_filepath)

NameError: name 'output_dir' is not defined

In [19]:
# Random input data
# x = torch.randn(1, 20, 576).cpu().numpy()
x = np.random.rand(20, 576).astype(np.float32)
x = x.reshape(1, 20, 576)

print(x.shape)

# compute ONNX Runtime output prediction
start_time = time.time()
ort_inputs = {ort_session.get_inputs()[0].name: x}
ort_outs = ort_session.run(None, ort_inputs)
print(f'El tiempo de inferencia con onnx es {(time.time()-start_time)*1000} ms')
print(ort_outs[0].shape)
print(ort_outs[0].dtype)
print(ort_outs[0][:5])

# Pytorch inference
start_time = time.time()
snr = net_snr.predict(x)
print(f'El tiempo de inferencia con pytorch es {(time.time()-start_time)*1000} ms')
snr = to_numpy(snr.squeeze())
print(snr.shape)
print(snr.dtype)
print(snr[:5])

(1, 20, 576)
El tiempo de inferencia con onnx es 14.790773391723633 ms
(1, 20, 512)
float32
[[[0.49885446 0.52871704 0.53973055 ... 0.08877164 0.08362639 0.07104588]
  [0.41957968 0.40724564 0.41924995 ... 0.06188631 0.06023979 0.05153894]
  [0.22204816 0.19764107 0.20025182 ... 0.01314485 0.01375854 0.0129447 ]
  ...
  [0.08107996 0.0683018  0.05636996 ... 0.01284981 0.0127514  0.0132134 ]
  [0.09233993 0.07748842 0.06413448 ... 0.01548332 0.01528025 0.01600277]
  [0.08012646 0.06793845 0.05608249 ... 0.0107944  0.01060319 0.01095754]]]
El tiempo de inferencia con pytorch es 33.486127853393555 ms
(20, 512)
float32
[[0.4988556  0.5287179  0.5397316  ... 0.08877156 0.08362627 0.07104579]
 [0.41958085 0.40724632 0.4192511  ... 0.0618862  0.06023962 0.05153874]
 [0.2220483  0.19764116 0.2002517  ... 0.01314485 0.01375857 0.01294472]
 [0.15745792 0.14087775 0.13609344 ... 0.00732281 0.00772638 0.00732855]
 [0.11697651 0.10293693 0.09070213 ... 0.0071165  0.0077161  0.00744494]]


In [ ]:
# Parámetros
fs=8000
B=32
w=[0.040]
m=0.01
nfft=1024
gmin = 0.0562
min_windows = 7
max_windows = 23
diezmation_factor = 2

# Cálculo de los filtros
N =[int(wi * fs) for wi in w]
F = int(nfft/2)
fb_time = time.time()
fb = fb_etsi(F, B, fs)
print(f'El tiempo de cálculo de los filtros es {(time.time() - fb_time)*1000} ms')
dct_time = time.time()
dct = f_base_dct(B)
print(f'El tiempo de cálculo de las bases dct es {(time.time() - dct_time)*1000} ms')

# Media y desviación para la normalización
file = workspace_dir + 'data/model/fe1_norm1.pkl'
mu, std = read_pkl(file)


# Ventana de hamming
hamming_win = np.hamming(fs * w[0])

# Variables globales para el cálculo de los tiempos
accum_offset = 0
accum_preemphasis = 0
accum_windowing = 0
accum_fft = 0
accum_log = 0
accum_fb_mfcc = 0
accum_norm = 0 
accum_inf = 0
diff_acum = 0
diff_inf_acum = 0


## SELECCIÓN DE AUDIO
# AUDIOS 16K
# x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/livekit/audio_received.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner_2023/afterburner16k/data/audio/callcenter_movistar/AUDIOS/Audio-AudioModule_708351_AudioChannel_22403522_19-May-2023_15.53.49.097.wav']

# AUDIOS 8K
x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/11-CH0_C01_construction_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/36635c2b-ca8a-5d7c-abb6-42981f7613e9.wav']

print(f'El audio elegido es {x_test[0]}')

if(reduced_net):
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16_opt.wav')
    print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
else:
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16_opt_onx.wav')
    # print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
    print(f'El audio mejorado es {output_enh_int} de tipo int16')



print(f'\n|-----------------------------INITIAL PARAMETERS-----------------------------|')
audio, fs = read_audio(x_test[0])
print(f'  Frecuencia de muestreo: {fs} Hz')
print(f'  Duración del audio: {len(audio)/fs} s - {len(audio)} samples')
frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
print(f'  Tamaño de frame: {frame_samples} samples')
shift_size = m  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
print(f'  Desplazamiento: {shift_samples} samples')
window_size = w[0]  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640
print(f'  Tamaño de ventana: {window_samples} samples')
window_inference_min = int((min_windows + (window_size/frame_size)-1) * frame_samples)
window_inference_max = int((max_windows + (window_size/frame_size)-1) * frame_samples)
print(f'  Buffer progresivo: {window_inference_min} - {window_inference_max} samples')
buffer_frame = np.zeros(0)  # Buffer de ventana recibida
Xfft_windows_list = []
print(f'|----------------------------------------------------------------------------|\n')

it = 0
snr_frame_mask = np.ones((512,min_windows)) # Inicializado con la duración de la ventana de inferencia
yenh = np.zeros(len(audio)) # Inicializado con la duración del audio original

# CALCULO DE LA MÁSCARA SNR 
for n_frame in range(int((len(audio)/fs)*100)):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]

    # print(f'\n{n_frame} frame de {len(frame)} --> {frame[:10]}')
    buffer_frame = np.concatenate([buffer_frame, frame])
    
    start_time = time.time()

    if len(buffer_frame) >= window_samples:
        work_window = buffer_frame[:window_samples]

        start_fft = time.time()
        Xfft = window_fft(work_window, hamming_win, nfft)
        accum_fft += (time.time()-start_fft)

        start_log = time.time()
        log_psd_Xfft = log_psd(Xfft)
        accum_log += (time.time()-start_log)

        start_fb = time.time()
        fb_windows = frame_fb_mfcc_2(Xfft, fb, dct)
        accum_fb_mfcc += (time.time()-start_fb)

        windows_concat = np.concatenate((log_psd_Xfft,fb_windows))
        Xfft_windows_list.append(windows_concat)

        if n_frame < max_windows:   
                if n_frame >= min_windows:
                    # print("Reached MIN WINDOW --> STRATING INFERENCE")
                    transformed_windows = np.vstack(Xfft_windows_list)

                    start_prof = time.time()
                    if(n_frame % diezmation_factor == 0):
                        if transformed_windows.shape[0] < 20:
                            snr_frame_mask = net_eval(transformed_windows)
                            print(f'Time: {(time.time()-start_prof)*1000} ms')
                        else:
                            transformed_windows = transformed_windows.reshape(1, 20, 576)
                            ort_inputs = {ort_session.get_inputs()[0].name: transformed_windows}
                            snr_frame_mask = ort_session.run(None, ort_inputs)[0]
                            print(f'Time onnx {n_frame}: {(time.time()-start_prof)*1000} ms')
                            

                        snr_frame_mask = snr_frame_mask.T
                    accum_inf += (time.time()-start_prof)

        # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
        else:
            Xfft_windows_list.pop(0)
            transformed_windows = np.vstack(Xfft_windows_list)
            start_prof = time.time()
            # Process inference with factor 2 diezmation
            if(n_frame % diezmation_factor == 0):
                transformed_windows = transformed_windows.reshape(1, 20, 576)
                ort_inputs = {ort_session.get_inputs()[0].name: transformed_windows}
                snr_frame_mask = ort_session.run(None, ort_inputs)[0]
                snr_frame_mask = snr_frame_mask.squeeze().T
                print(f'Time onnx {n_frame}: {(time.time()-start_prof)*1000} ms')
            accum_inf += (time.time()-start_prof)

        buffer_frame = buffer_frame[shift_samples:]

        # Aqui haría la evaluacion con la máscara pertinente (para las primeras 3 ventanas sin máscara calculada)
        # cnt = int(n_frame - w[0]/m) Ajustar al tamaño de la ventana
        cnt = n_frame - 3
        # print(f'EVALUATION OF WINDOW {cnt}')
        x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples

        xenh, filt = noiseReduction(x, snr_frame_mask[:,-1], fs, window_samples, shift_samples, nfft, gmin)

        slice_size = min(len(yenh) - cnt * shift_samples, window_samples)

        yenh[cnt * shift_samples : cnt * shift_samples + slice_size] += xenh[0:slice_size]


    end_time = time.time()
    diff_acum += end_time - start_time
    # print(f'Tiempo de procesamiento del frame {n_frame} completo es de {(end_time-start_time)*1000} ms')

print(f'\n|-----------------------------FINAL TIME STATS-------------------------------|')
print(f'  El tiempo medio de procesamiento de la FFT es de {(accum_fft/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la escala log es de {(accum_log/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la FB MFCC es de {(accum_fb_mfcc/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la inferencia es de {(accum_inf/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento total es de {(diff_acum/n_frame)*1000} ms')
print(f'|----------------------------------------------------------------------------|\n')


yenh = yenh/3     # 4 because in OverLapAdd we sum 4 times the frame

yenh = np.array(yenh*(2 ** 15), dtype=np.int16)     # set int16 wav format

wavfile.write(output_enh_int,fs,yenh)



El tiempo de cálculo de los filtros es 1.3909339904785156 ms
El tiempo de cálculo de las bases dct es 0.96893310546875 ms
El audio elegido es /home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav
El audio mejorado es /home/adelval/BTS/TFM/audios/enh/5-CH0_C01_stadium_15dB_f2f_enh_HM_2_8k_7to23w_40ms_s10_int16_opt_onx.wav de tipo int16

|-----------------------------INITIAL PARAMETERS-----------------------------|
  Frecuencia de muestreo: 8000 Hz
  Duración del audio: 3.374125 s - 26993 samples
  Tamaño de frame: 80 samples
  Desplazamiento: 80 samples
  Tamaño de ventana: 320 samples
  Buffer progresivo: 800 - 2080 samples
|----------------------------------------------------------------------------|

snr (512,)
snr (512,)
snr (512,)
snr (512,)
snr (512,)
Time: 48.48504066467285 ms
snr (512,)
snr (512,)
Time: 22.1560001373291 ms
snr (512,)
snr (512,)
Time: 22.78900146484375 ms
snr (512,)
snr (512,)
Time: 37.89997100830078 ms
snr (512,)
snr (512,)
Time: 

ValueError: operands could not be broadcast together with shapes (513,) (20,) 